# CBIS-DDSM — Exploratory Data Analysis

Covers both the **images** and the **clinical metadata**. The metadata exploration (density,
shape, margins, assessment, subtlety, calcification type/distribution) and how each relates to
malignancy is what motivates the **image + metadata multimodal** direction.

Set `DATA_DIR` (the folder containing `csv/` and `jpeg/`) and Run All.

In [ ]:
# ---------------- CONFIG ----------------
DATA_DIR = "/path/to/archive (1)"   # <-- EDIT: folder containing csv/ and jpeg/
N_SAMPLE_IMAGES = 6                  # sample mammograms shown per class
N_DIM_SAMPLE    = 150               # images sampled to estimate dimension stats
# ----------------------------------------
import os, re, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
random.seed(0)
CSV_DIR = os.path.join(DATA_DIR, "csv")
def norm(df): df = df.copy(); df.columns = [c.strip().lower() for c in df.columns]; return df
def getcol(df, *names):
    for n in names:
        if n in df.columns: return n
    return None

## 1. Load the data and inspect structure

In [ ]:
mass_tr = norm(pd.read_csv(os.path.join(CSV_DIR, "mass_case_description_train_set.csv")))
mass_te = norm(pd.read_csv(os.path.join(CSV_DIR, "mass_case_description_test_set.csv")))
calc_tr = norm(pd.read_csv(os.path.join(CSV_DIR, "calc_case_description_train_set.csv")))
calc_te = norm(pd.read_csv(os.path.join(CSV_DIR, "calc_case_description_test_set.csv")))
dicom   = norm(pd.read_csv(os.path.join(CSV_DIR, "dicom_info.csv")))

mass = pd.concat([mass_tr.assign(split="train"), mass_te.assign(split="test")], ignore_index=True)
calc = pd.concat([calc_tr.assign(split="train"), calc_te.assign(split="test")], ignore_index=True)
mass["label"] = (mass["pathology"].str.upper() == "MALIGNANT").astype(int)
calc["label"] = (calc["pathology"].str.upper() == "MALIGNANT").astype(int)

for name, df in [("mass", mass), ("calc", calc), ("dicom_info", dicom)]:
    print(f"{name:11s} shape={df.shape}")
print("\nMass columns:\n", list(mass.columns))
print("\nCalc columns:\n", list(calc.columns))
mass.head(3)

## 2. Missing values

In [ ]:
print("Mass missing values:\n", mass.isnull().sum()[mass.isnull().sum() > 0], "\n")
print("Calc missing values:\n", calc.isnull().sum()[calc.isnull().sum() > 0])

## 3. Class distribution (pathology)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (name, df) in zip(ax, [("Mass", mass), ("Calc", calc)]):
    vc = df["pathology"].value_counts()
    a.bar(vc.index, vc.values, color="#3b6ea5")
    a.set_title(f"{name} cases — pathology"); a.tick_params(axis="x", rotation=20)
    for i, v in enumerate(vc.values): a.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout(); plt.show()

print("Malignant proportion (binary):")
print(f"  Mass: {mass['label'].mean():.3f}   Calc: {calc['label'].mean():.3f}")
print("\nBy split (mass):"); print(mass.groupby("split")["label"].agg(["count","mean"]))

## 4. Clinical-metadata distributions\nThe fields that a multimodal model would fuse with the image.

In [ ]:
def bar_counts(df, col, title, ax, top=None):
    if col is None: ax.axis("off"); return
    vc = df[col].astype(str).value_counts()
    if top: vc = vc.head(top)
    ax.bar(range(len(vc)), vc.values, color="#5a8f69")
    ax.set_xticks(range(len(vc))); ax.set_xticklabels(vc.index, rotation=35, ha="right", fontsize=8)
    ax.set_title(title, fontsize=10)

dens_m = getcol(mass, "breast_density", "breast density")
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
bar_counts(mass, dens_m, "Mass: breast density", ax[0,0])
bar_counts(mass, "assessment", "Mass: assessment (BI-RADS)", ax[0,1])
bar_counts(mass, "subtlety", "Mass: subtlety", ax[0,2])
bar_counts(mass, getcol(mass,"mass shape"), "Mass: shape", ax[1,0], top=10)
bar_counts(mass, getcol(mass,"mass margins"), "Mass: margins", ax[1,1], top=10)
bar_counts(mass, "image view", "Mass: view", ax[1,2])
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
bar_counts(calc, getcol(calc,"calc type"), "Calc: type", ax[0], top=10)
bar_counts(calc, getcol(calc,"calc distribution"), "Calc: distribution", ax[1], top=10)
bar_counts(calc, "assessment", "Calc: assessment (BI-RADS)", ax[2])
plt.tight_layout(); plt.show()

## 5. Malignancy rate by metadata field\nDoes each clinical feature carry signal? (Strong motivation for multimodal fusion.)

In [ ]:
def malig_rate(df, col, title, ax, top=None):
    if col is None: ax.axis("off"); return
    g = df.groupby(df[col].astype(str))["label"].agg(["mean","count"])
    g = g[g["count"] >= 10].sort_values("mean")
    if top: g = g.tail(top)
    ax.barh(range(len(g)), g["mean"].values, color="#b5651d")
    ax.set_yticks(range(len(g))); ax.set_yticklabels(g.index, fontsize=8)
    ax.set_xlabel("malignant fraction"); ax.set_title(title, fontsize=10); ax.set_xlim(0,1)

fig, ax = plt.subplots(2, 2, figsize=(13, 9))
malig_rate(mass, "assessment", "Mass: malignancy by assessment", ax[0,0])
malig_rate(mass, dens_m, "Mass: malignancy by density", ax[0,1])
malig_rate(mass, getcol(mass,"mass margins"), "Mass: malignancy by margins", ax[1,0], top=10)
malig_rate(mass, getcol(mass,"mass shape"), "Mass: malignancy by shape", ax[1,1], top=10)
plt.tight_layout(); plt.show()

## 6. Linking labels to full-mammogram JPEGs

In [ ]:
def local_jpeg(p):
    p = str(p).replace("\\", "/")
    rel = p.split("jpeg/", 1)[1] if "jpeg/" in p else os.path.basename(p)
    return os.path.join(DATA_DIR, "jpeg", rel)

full = dicom[dicom["seriesdescription"].astype(str).str.contains("full mammogram", case=False, na=False)].copy()
pat = full["patientid"].astype(str).str.extract(
    r"(?P<cohort>Mass|Calc)-(?P<split>Training|Test)_(?P<patient_id>P_\d+)_(?P<side>LEFT|RIGHT)_(?P<view>CC|MLO)")
full = pd.concat([full.reset_index(drop=True), pat.reset_index(drop=True)], axis=1)
full["jpeg"] = full["image_path"].apply(local_jpeg)

lab = (mass.groupby(["patient_id","left or right breast","image view"], as_index=False)["label"].max()
          .rename(columns={"left or right breast":"side","image view":"view"}))
mass_imgs = (full[full["cohort"]=="Mass"].merge(lab, on=["patient_id","side","view"], how="inner"))
mass_imgs = mass_imgs[mass_imgs["jpeg"].apply(os.path.exists)].drop_duplicates("jpeg")
print(f"Full mammograms in dicom_info: {len(full)}")
print(f"Mass images linked with labels: {len(mass_imgs)}")
print(mass_imgs["label"].value_counts().rename({0:"not-malignant",1:"malignant"}))
if len(mass_imgs) == 0:
    print("[!] Linking returned 0 — paste me: ", list(dicom.columns),
          full[["patientid","seriesdescription","image_path"]].head(3).to_dict("records"))

## 7. Sample mammograms per class

In [ ]:
def show_grid(df_sub, title):
    df_sub = df_sub.sample(min(N_SAMPLE_IMAGES, len(df_sub)), random_state=1)
    fig, ax = plt.subplots(1, len(df_sub), figsize=(3*len(df_sub), 3.2))
    if len(df_sub) == 1: ax = [ax]
    for a, (_, r) in zip(ax, df_sub.iterrows()):
        a.imshow(Image.open(r["jpeg"]).convert("L"), cmap="gray"); a.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

if len(mass_imgs):
    show_grid(mass_imgs[mass_imgs["label"]==1], "Sample MALIGNANT mammograms")
    show_grid(mass_imgs[mass_imgs["label"]==0], "Sample NOT-MALIGNANT mammograms")

## 8. Full mammogram + cropped lesion + ROI mask\nThe ROI masks are the ground truth your attention-supervision objective depends on.

In [ ]:
if len(mass_imgs):
    ex = mass_imgs.iloc[0]
    base = f"Mass-{'Training' if ex['split']=='train' else 'Test'}_{ex['patient_id']}_{ex['side']}_{ex['view']}"
    def first_jpeg(desc):
        sub = dicom[(dicom["patientid"].astype(str).str.startswith(base)) &
                    (dicom["seriesdescription"].astype(str).str.contains(desc, case=False, na=False))]
        for _, r in sub.iterrows():
            p = local_jpeg(r["image_path"])
            if os.path.exists(p): return p
        return None
    panels = [("Full mammogram", ex["jpeg"]),
              ("Cropped lesion", first_jpeg("cropped")),
              ("ROI mask", first_jpeg("ROI mask"))]
    fig, ax = plt.subplots(1, 3, figsize=(11, 4))
    for a, (t, p) in zip(ax, panels):
        if p: a.imshow(Image.open(p).convert("L"), cmap="gray")
        a.set_title(t); a.axis("off")
    plt.suptitle(f"Example: {base}"); plt.tight_layout(); plt.show()

## 9. Image dimension distribution (sampled)

In [ ]:
if len(mass_imgs):
    sizes = []
    for p in mass_imgs["jpeg"].sample(min(N_DIM_SAMPLE, len(mass_imgs)), random_state=2):
        with Image.open(p) as im: sizes.append(im.size)  # (w, h)
    sizes = np.array(sizes)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].hist(sizes[:,0], bins=30, color="#3b6ea5"); ax[0].set_title("Width (px)")
    ax[1].hist(sizes[:,1], bins=30, color="#5a8f69"); ax[1].set_title("Height (px)")
    plt.tight_layout(); plt.show()
    print(f"Width  min/median/max: {sizes[:,0].min()}/{int(np.median(sizes[:,0]))}/{sizes[:,0].max()}")
    print(f"Height min/median/max: {sizes[:,1].min()}/{int(np.median(sizes[:,1]))}/{sizes[:,1].max()}")

## 10. EDA summary (fill in your observations)

- **Class balance:** _e.g. proportion malignant in mass vs calc; note the imbalance to handle._
- **Most predictive metadata:** _which assessment / margin / shape categories carry the strongest malignancy signal — supports multimodal fusion._
- **Image sizes:** _range of resolutions; justifies resizing to 224×224._
- **Data quality:** _any missing fields or unlinked images._
- **Implication for modelling:** _binary vs 3-class choice; imbalance handling; metadata to include._
